In [62]:
import pandas as pd

In [79]:
# ==========================================
# 1. file path
# ==========================================

file1 = "nist_Ar.xlsx"   # Wavelength + Species
file2 = "importance_wavelenth.xlsx"   #  Wavelength 
file3 = "other_wavelenght.xlsx"

In [64]:
df1 = pd.read_excel(file1)
df1 = df1.drop(['Intensity', 'Reference'], axis=1, errors='ignore')
df1['Air'] = df1['Air'].astype(float) / 10 
df1

,Air,Spectrum
0,242.0456,Ar II
1,251.6789,Ar II
2,253.4709,Ar II
3,256.2087,Ar II
4,289.1612,Ar II
...,...,...
254,1694.0580,Ar I
255,2061.6230,Ar I
256,2098.6110,Ar I
257,2313.3200,Ar I


In [65]:
df2 = pd.read_excel(file2)
df2 = df2.drop(['Unnamed: 0', 'Importance'], axis=1, errors='ignore')
df2 = df2.drop(index=0, errors='ignore')
df2['Feature'] = pd.to_numeric(df2['Feature'], errors="coerce") 
df2

,Feature
1,854.716602
2,887.258212
3,355.266077
4,744.595702
5,227.527369
6,751.789051
7,762.606641
8,800.212452
9,746.658500
10,751.448194


In [87]:
df3 = pd.read_excel(file3)
# df2 = df2.drop(index=0, errors='ignore')
df3['Air'] = pd.to_numeric(df3['Air'], errors="coerce") 
df3

,Air,Spectrum
0,202.11,NO (β system)
1,210.36,NO (β system)
2,203.07,NO (γ system)
3,205.28,NO (γ system)
4,214.91,NO (γ system)
...,...,...
91,866.79,Ar I
92,912.30,Ar I
93,922.45,Ar I
94,656.20,Hα


In [88]:
df1 = pd.concat([df3, df1], ignore_index=True)
df1

,Air,Spectrum
0,202.110,NO (β system)
1,210.360,NO (β system)
2,203.070,NO (γ system)
3,205.280,NO (γ system)
4,214.910,NO (γ system)
...,...,...
350,1694.058,Ar I
351,2061.623,Ar I
352,2098.611,Ar I
353,2313.320,Ar I


In [90]:
wavelength_col_1 = "Air"
species_col = "Spectrum"
wavelength_col_2 = "Feature"

In [91]:
df2['_original_order'] = range(len(df2))
df2

,Feature,_original_order
1,854.716602,0
2,887.258212,1
3,355.266077,2
4,744.595702,3
5,227.527369,4
6,751.789051,5
7,762.606641,6
8,800.212452,7
9,746.658500,8
10,751.448194,9


In [ ]:

df1_sorted = df1.sort_values('Air').copy()
df2_sorted = df2.sort_values('Feature').copy()

In [93]:
result = pd.merge_asof(
    df2_sorted,
    df1_sorted,
    left_on='Feature',
    right_on='Air',
    direction='nearest'
)

In [ ]:
# -------------------------
# -------------------------
result = result.sort_values('_original_order')

# -------------------------
# -------------------------
result = result[
    ['Feature', 'Air', 'Spectrum']
].copy()


result.columns = [
    'Best_Wavelength_FeatureDataset2',
    'Nist_Wavelength_Dataset1',
    'NistSpecies'
]

print(result)

result.to_excel(
    'wavelength_species_matching.xlsx',
    index=False
)

    Best_Wavelength_FeatureDataset2  Nist_Wavelength_Dataset1    NistSpecies
28                       854.716602                  856.7800             NI
30                       887.258212                  884.9910           Ar I
2                        355.266077                  354.5845          Ar II
7                        744.595702                  743.5368           Ar I
0                        227.527369                  226.2800  NO (γ system)
11                       751.789051                  751.4700           Ar I
13                       762.606641                  763.5100           Ar I
20                       800.212452                  800.6157           Ar I
8                        746.658500                  743.5368           Ar I
10                       751.448194                  751.4652           Ar I
9                        749.057470                  750.3869           Ar I
27                       851.444149                  852.1400           Ar I

In [95]:
result

,Best_Wavelength_FeatureDataset2,Nist_Wavelength_Dataset1,NistSpecies
28,854.716602,856.7800,NI
30,887.258212,884.9910,Ar I
2,355.266077,354.5845,Ar II
7,744.595702,743.5368,Ar I
0,227.527369,226.2800,NO (γ system)
11,751.789051,751.4700,Ar I
13,762.606641,763.5100,Ar I
20,800.212452,800.6157,Ar I
8,746.658500,743.5368,Ar I
10,751.448194,751.4652,Ar I


In [96]:
import pandas as pd
import docx

from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT


# ==========================================
# 1. Data
# ==========================================

dataimportance = result.copy()


# ==========================================
# 2. Create Word document
# ==========================================

doc = docx.Document()


# ==========================================
# 3. Title
# ==========================================

title = doc.add_paragraph()

title.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = title.add_run("Feature Importance Analysis")

run.font.size = Pt(16)
run.font.bold = True


# ==========================================
# 4. Get column names automatically
# ==========================================

headers = ['Rank'] + list(dataimportance.columns)


# ==========================================
# 5. Create table automatically
# ==========================================

table = doc.add_table(
    rows=len(dataimportance) + 1,
    cols=len(headers)
)

table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.style = 'Table Grid'


# ==========================================
# 6. Header
# ==========================================

for idx, text in enumerate(headers):

    cell = table.rows[0].cells[idx]

    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    r = p.add_run(str(text))
    r.font.bold = True


# ==========================================
# 7. Fill table automatically
# ==========================================

for i, (_, row) in enumerate(
    dataimportance.iterrows(),
    start=1
):

    # -------------------------
    # No
    # -------------------------

    cell = table.rows[i].cells[0]

    cell.text = str(i)

    cell.paragraphs[0].alignment = (
        WD_ALIGN_PARAGRAPH.CENTER
    )


    # -------------------------
    # Dataset columns
    # -------------------------

    for j, column in enumerate(
        dataimportance.columns,
        start=1
    ):

        cell = table.rows[i].cells[j]

        value = row[column]

        # Numeric values
        if pd.api.types.is_number(value):

            cell.text = f"{value:.6f}"

        # Text values
        else:

            cell.text = str(value)

        cell.paragraphs[0].alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )


# ==========================================
# 8. Save
# ==========================================

doc.save(r"finde_nist.docx")

print("Done....")

Done....
